# Phase 5: Legal Statutes Prediction

 _This is submission 1 where we fine tune our phase 1 Decoder-only transformer with LORMA adaptation for downstream task of Legal Statutes Prediction._

## 1.Login and GPU check

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 2. Dataloading and Pre-processing

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer


HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone" 

# --- 1. Data Loading ---
print("Loading datasets from Hugging Face Hub...")

try:
    # Explicitly specify 'phase_4' here
    hf_dataset = load_dataset(HF_DATASET_REPO, 'phase_5')
    print(f"Successfully loaded splits: {list(hf_dataset.keys())}")
except Exception as e:
    print(f"Error loading dataset: {e}")

def standardize_labels(example, is_test=False):
    if not is_test:
        if 'pred' in example and 'label' not in example:
             example['label'] = example['pred']
    return example

# Apply standard formatting
for split in hf_dataset.keys():
    is_test_split = (split == 'test')
    hf_dataset[split] = hf_dataset[split].map(lambda x: standardize_labels(x, is_test=is_test_split))

# Assuming you have N total distinct statutes in Phase 5
NUM_STATUTES = 100 

def multi_hot_encode(example):
    # Convert list of statute IDs to a binary vector of size NUM_STATUTES
    labels = np.zeros(NUM_STATUTES, dtype=np.float32)
    if "labels" in example and example["labels"] is not None:
        for statute_id in example["labels"]:
            labels[statute_id] = 1.0
        example['labels'] = labels
    return example

hf_dataset = hf_dataset.map(multi_hot_encode)
print("Data loading complete.")

In [ ]:
# --- 2. Preprocessing & Strategic Truncation ---

from tokenizers import Tokenizer
from transformers import PreTrainedTokenizerFast

BPE_FILE = "/kaggle/input/datasets/vaibhavim081203/phase3feed/cs781_phase1/legal_bpe.json"

# Load backend tokenizer directly
backend_tok = Tokenizer.from_file(BPE_FILE)

# Wrap into Hugging Face PreTrainedTokenizerFast
tokenizer = PreTrainedTokenizerFast(tokenizer_object=backend_tok)

# Define special tokens according to your Phase 1 setup
tokenizer.pad_token = "[PAD]" if "[PAD]" in tokenizer.get_vocab() else tokenizer.eos_token
tokenizer.cls_token = "[CLS]" if "[CLS]" in tokenizer.get_vocab() else "<s>"
tokenizer.sep_token = "[SEP]" if "[SEP]" in tokenizer.get_vocab() else "</s>"

print(f"Tokenizer loaded successfully! Vocab size: {len(tokenizer)}")

# Safe fallback for special tokens
START_ID = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else (tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0)
END_ID = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

MAX_LENGTH = 1024 #
STRIDE = 800      # Overlap of 224 tokens to keep sentences intact
MAX_CHUNKS_PER_DOC= 12

def sliding_window_chunking(examples):
    tokenized_inputs = tokenizer(
        examples["text"],
        padding=False,
        truncation=False,
        add_special_tokens=False
    )
    
    input_ids_list = []
    attention_mask_list = []
    
    for input_ids in tokenized_inputs["input_ids"]:
        chunks = []
        masks = []
        
        # Slide over the long document
        for i in range(0, len(input_ids), STRIDE):
            chunk = input_ids[i : i + (MAX_LENGTH - 2)]
            # Add Special Tokens safely without injecting None
            final_ids = [START_ID] + chunk + [END_ID]
            attention_mask = [1] * len(final_ids)
            
            chunks.append(final_ids)
            masks.append(attention_mask)
            
        # Enforce the cap safely using list slicing
        # If len(chunks) <= 12, it does nothing. If > 12, it truncates the tail.
        input_ids_list.append(chunks[:MAX_CHUNKS_PER_DOC])
        attention_mask_list.append(masks[:MAX_CHUNKS_PER_DOC])
        
    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list
    }

tokenized_datasets = hf_dataset.map(sliding_window_chunking, batched=True, remove_columns=["text"])

In [ ]:
import torch
import numpy as np

print("Calculating class frequencies for pos_weight...")
num_train_cases = len(hf_dataset["train"])
statute_counts = np.zeros(NUM_STATUTES)

for example in hf_dataset["train"]:
    # Using 'pred' or 'labels' depending on how it's named in your phase 5 dataset
    if "pred" in example and example["pred"] is not None:
        for statute_id in example["pred"]:
            statute_counts[statute_id] += 1

epsilon = 1e-5
pos_weights = (num_train_cases - statute_counts) / (statute_counts + epsilon)
pos_weights = np.clip(pos_weights, a_min=1.0, a_max=50.0)

pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float32)
print("pos_weight tensor created successfully!")

## 3. Model Initialization and LORMA adaptations

In [ ]:
import math
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F

NEG_INF = -65504.0   # most negative finite fp16 value: "masked" without risking NaN from -inf arithmetic


@dataclass
class LegalLMConfig:
    vocab_size: int = 32000      # Custom BPE tokenizer vocabulary
    d_model: int = 768           # Model hidden dimension
    n_heads: int = 12            # Number of attention heads
    n_layers: int = 10           # Stacked layer depth (~95M parameters)
    max_seq_len: int = 2048      # Maximum context window length
    dropout: float = 0.1         # Dropout probability
    bias: bool = True            # Enable bias in Linear layers


def compute_alibi_slopes(n_heads: int) -> torch.Tensor:
    """
    Geometric slopes for Attention with Linear Biases (ALiBi).
    Reference: Press et al., "Train Short, Test Long: Attention with Linear Biases"
    (matches the reference implementation, including for head counts that are not a power of two, e.g. 12).
    """
    n = 2 ** math.floor(math.log2(n_heads))
    base = 2 ** (-(2 ** -(math.log2(n) - 3)))
    slopes = [base * (base ** i) for i in range(n)]

    if n < n_heads:
        extra_base = 2 ** (-(2 ** -(math.log2(2 * n) - 3)))
        # reference: get_slopes(2n)[0::2][: n_heads - n]  ->  extra_base ** (2i + 1)
        extra_slopes = [extra_base ** (2 * i + 1) for i in range(n_heads - n)]
        slopes.extend(extra_slopes)

    return torch.tensor(slopes, dtype=torch.float32)


class ALiBiCausalSelfAttention(nn.Module):
    """
    Causal self-attention with ALiBi, computed with F.scaled_dot_product_attention (fused kernel):
    the (T x T) attention map is never materialised in Python, and the causal mask + ALiBi penalty are
    handed to the kernel as ONE additive bias that is built once per forward pass (see CustomLegalLM._alibi_bias).
    Supports a key/value cache for fast generation.
    """
    def __init__(self, config: LegalLMConfig):
        super().__init__()
        assert config.d_model % config.n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = config.n_heads
        self.d_head = config.d_model // config.n_heads
        self.dropout_p = config.dropout

        self.c_attn = nn.Linear(config.d_model, 3 * config.d_model, bias=config.bias)   # fused Q,K,V
        self.c_proj = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x, attn_bias, past_kv=None, use_cache=False):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)   # (B, H, T, dh)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        # attn_bias is a {dtype: tensor} holder: the fp16 copy is made ONCE per forward pass and shared by all layers
        # (so autograd keeps a single copy for backward instead of one per layer)
        bias = attn_bias.get(q.dtype)
        if bias is None:
            bias = attn_bias[q.dtype] = attn_bias[torch.float32].to(q.dtype)
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=bias,
            dropout_p=self.dropout_p if self.training else 0.0,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y, present


class FeedForward(nn.Module):
    def __init__(self, config: LegalLMConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.d_model, 4 * config.d_model, bias=config.bias),
            nn.GELU(),
            nn.Linear(4 * config.d_model, config.d_model, bias=config.bias),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, config: LegalLMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = ALiBiCausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.mlp = FeedForward(config)

    def forward(self, x, attn_bias, past_kv=None, use_cache=False):
        a, present = self.attn(self.ln1(x), attn_bias, past_kv, use_cache)
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return x, present


class CustomLegalLM(nn.Module):
    def __init__(self, config: LegalLMConfig):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.d_model),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)]),
            ln_f=nn.LayerNorm(config.d_model),
        ))
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        # Weight tying: input embedding == output projection
        self.transformer.wte.weight = self.lm_head.weight

        # ALiBi slopes live on the model (not per layer); non-persistent -> not stored in checkpoints
        self.register_buffer("alibi_slopes", compute_alibi_slopes(config.n_heads).view(1, config.n_heads, 1, 1),
                             persistent=False)
        self._bias_key = None
        self._bias_val = None

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)

    def get_num_params(self) -> int:
        """Trainable parameters (the tied embedding / output matrix is counted once)."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def _alibi_bias(self, T: int, S: int, device) -> dict:
        """
        Additive attention bias (returned as {dtype: tensor}, fp32 entry of shape (1, H, T, S)): -slope * distance for past keys,
        a large negative value for future keys (causal mask). The T queries are the LAST T of the S keys,
        so the same function serves training (T == S) and cached decoding (T == 1, S = cache length + 1).
        Cached for repeated identical shapes (every training step).
        """
        key = (T, S, str(device))
        if self._bias_key == key:
            return self._bias_val
        q_pos = torch.arange(S - T, S, device=device).view(T, 1)
        k_pos = torch.arange(S, device=device).view(1, S)
        dist = (q_pos - k_pos).view(1, 1, T, S)
        bias = -self.alibi_slopes.to(device).float() * dist.clamp(min=0).float()
        bias = bias.masked_fill(dist < 0, NEG_INF)
        self._bias_key, self._bias_val = key, {torch.float32: bias}
        return self._bias_val

    def _run(self, input_ids, past_kvs=None, use_cache=False):
        B, T = input_ids.size()
        past_len = 0 if past_kvs is None else past_kvs[0][0].size(2)
        bias = self._alibi_bias(T, past_len + T, input_ids.device)

        x = self.transformer.drop(self.transformer.wte(input_ids))   # ALiBi supplies the position information
        presents = []
        for i, block in enumerate(self.transformer.h):
            x, present = block(x, bias, None if past_kvs is None else past_kvs[i], use_cache)
            presents.append(present)
        return self.transformer.ln_f(x), presents

    def forward(self, input_ids, targets=None):
        x, _ = self._run(input_ids)
        if targets is not None:
            logits = self.lm_head(x)                                  # full-sequence logits for the loss
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            logits = self.lm_head(x[:, [-1], :])                      # inference: last position only
            loss = None
        return logits, loss

    def forward_cached(self, input_ids, past_kvs=None):
        """Incremental forward: returns (last-position logits (B,1,V), new key/value cache)."""
        x, presents = self._run(input_ids, past_kvs, use_cache=True)
        return self.lm_head(x[:, [-1], :]), presents

    @staticmethod
    def _sample(logits, temperature, top_p):
        """logits: (B, V) -> next token ids (B, 1) with temperature + nucleus (top-p) sampling."""
        if temperature is None or temperature <= 0:
            return logits.argmax(dim=-1, keepdim=True)
        probs = F.softmax(logits.float() / temperature, dim=-1)
        if top_p is not None and top_p < 1.0:
            sorted_p, sorted_i = torch.sort(probs, descending=True, dim=-1)
            cum = sorted_p.cumsum(dim=-1)
            sorted_p = sorted_p.masked_fill((cum - sorted_p) > top_p, 0.0)   # always keeps the top token
            sorted_p = sorted_p / sorted_p.sum(dim=-1, keepdim=True)
            return sorted_i.gather(-1, torch.multinomial(sorted_p, 1))
        return torch.multinomial(probs, 1)

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=128, temperature=1.0, top_p=1.0, stop_fn=None, check_every=16):
        """
        Autoregressive generation with a KV cache (one prefill pass, then one token per step).
        input_ids: (B, T) with NO padding (use B=1 for prompts of different lengths).
        stop_fn(list_of_new_ids_of_first_sequence) -> bool lets the caller stop early (e.g. after 128 words).
        Returns the new token ids, shape (B, n_new).
        """
        self.eval()
        room = max(1, self.config.max_seq_len - max_new_tokens)
        input_ids = input_ids[:, -room:]                      # keep the END of an over-long prompt
        logits, past = self.forward_cached(input_ids, None)
        new_ids = []
        for t in range(max_new_tokens):
            nxt = self._sample(logits[:, -1, :], temperature, top_p)
            new_ids.append(nxt)
            if stop_fn is not None and (t + 1) % check_every == 0 and stop_fn(torch.cat(new_ids, 1)[0].tolist()):
                break
            if t + 1 < max_new_tokens:
                logits, past = self.forward_cached(nxt, past)
        return torch.cat(new_ids, dim=1)

In [ ]:
import torch
import torch.nn as nn
import math


class LegalLMForMultiLabelChunking(nn.Module):
    def __init__(self, config, num_labels=NUM_STATUTES, pos_weight_tensor=None):
        super().__init__()
        self.base_model = CustomLegalLM(config) 
        
        # Sever weight tying if it exists before deleting lm_head (Safe for DataParallel)
        if hasattr(self.base_model, 'lm_head') and hasattr(self.base_model.transformer, 'wte'):
            clean_weight = self.base_model.transformer.wte.weight.clone().detach()
            self.base_model.transformer.wte.weight = nn.Parameter(clean_weight)
            
        del self.base_model.lm_head 
        
        self.score = nn.Linear(config.d_model, num_labels, bias=False)
        
        # Freeze base model parameters initially for LORMA
        for param in self.base_model.parameters():
            param.requires_grad = False
            
        # --- Register pos_weight ---
        if pos_weight_tensor is not None:
            self.register_buffer('pos_weight', pos_weight_tensor)
        else:
            self.pos_weight = None

    def forward(self, input_ids, attention_mask, labels=None):
        batch_size, num_chunks, seq_len = input_ids.shape
        
        flat_input_ids = input_ids.view(batch_size * num_chunks, seq_len)
        flat_attention_mask = attention_mask.view(batch_size * num_chunks, seq_len)
        
        # Run through ALiBi transformer blocks
        x, _ = self.base_model._run(flat_input_ids)
        
        sequence_lengths = flat_attention_mask.sum(dim=1) - 1
        sequence_lengths = torch.clamp(sequence_lengths, min=0)
        
        batch_indices = torch.arange(x.shape[0], device=x.device)
        pooled_representation = x[batch_indices, sequence_lengths]
        
        chunk_logits = self.score(pooled_representation)
        chunk_logits = chunk_logits.view(batch_size, num_chunks, -1)
        
        chunk_active_mask = (flat_attention_mask.sum(dim=1) > 0).view(batch_size, num_chunks, 1)
         # --- THE FIX ---
        # Dynamically get the minimum safe value for the current dtype (float32 or float16)
        min_val = torch.finfo(chunk_logits.dtype).min
        chunk_logits = chunk_logits.masked_fill(~chunk_active_mask, min_val)
        
        document_logits, _ = torch.max(chunk_logits, dim=1)
        
        loss = None
        if labels is not None:
            # --- Apply pos_weight during loss calculation ---
            loss = nn.functional.binary_cross_entropy_with_logits(
                document_logits, 
                labels,
                pos_weight=self.pos_weight
            )
            
        return {"logits": document_logits, "loss": loss}




# --- 2. LORMA Adapter Injection ---

class LORMALinear(nn.Module):
    """
    Wraps an existing nn.Linear layer with Low-Rank parameters (A and B).
    LORMA typically initializes B to zero and A with a variance-scaling distribution,
    or utilizes SVD-based initialization depending on the specific LORMA variant.
    """
    def __init__(self, base_layer: nn.Linear, rank: int = 8, alpha: float = 16.0):
        super().__init__()
        self.base_layer = base_layer
        self.base_layer.weight.requires_grad = False
        if self.base_layer.bias is not None:
            self.base_layer.bias.requires_grad = False
            
        in_features = base_layer.in_features
        out_features = base_layer.out_features
        
        # Low rank matrices
        self.lora_A = nn.Parameter(torch.zeros(rank, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        self.scaling = alpha / rank
        
        # Initialization
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        base_out = self.base_layer(x)
        # $W = W_0 + B A$
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base_out + lora_out

def inject_lorma_into_model(model: nn.Module, rank: int = 8, alpha: float = 16.0):
    """Iterates through the transformer and replaces attention projections with LORMA."""
    for block in model.base_model.transformer.h:
        # Inject into fused Q,K,V projection
        block.attn.c_attn = LORMALinear(block.attn.c_attn, rank=rank, alpha=alpha)
        # Inject into output projection
        block.attn.c_proj = LORMALinear(block.attn.c_proj, rank=rank, alpha=alpha)
    
    # Ensure the classification head is trainable
    model.score.weight.requires_grad = True
    return model


# --- 3. Instantiation & Weight Loading ---

print("Initializing architecture...")
# Use the same configuration from Phase 1
config = LegalLMConfig(max_seq_len=1024) 
model = LegalLMForMultiLabelChunking(
    config, 
    num_labels=NUM_STATUTES, 
    pos_weight_tensor=pos_weight_tensor
)

print("Loading Phase 1 Pretrained Weights...")
checkpoint_path = "/kaggle/input/datasets/vaibhavim081203/phase3feed/cs781_phase1/checkpoints/best_model.pt" # Update path
state_dict = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

# Filter out lm_head weights since we deleted it
state_dict = {k: v for k, v in state_dict.items() if "lm_head" not in k}
model.base_model.load_state_dict(state_dict, strict=False)

print("Injecting LORMA adapters...")
model = inject_lorma_into_model(model, rank=8, alpha=16.0)

# Verify trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable Parameters: {trainable_params:,} ({(trainable_params/total_params)*100:.2f}%)")

## 4. Training loop and Evaluation

In [ ]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW  # <-- Import from PyTorch instead
from tqdm.auto import tqdm
import random
import numpy as np
import os

# --- 1. DataLoader & Padding Setup ---

PAD_TOKEN_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0 

def collate_fn_chunked(batch):
    max_chunks = max(len(item['input_ids']) for item in batch)
    max_len = max(max(len(chunk) for chunk in item['input_ids']) for item in batch)
    
    batch_input_ids = []
    batch_attention_mask = []
    labels, ids = [], []
    
    for item in batch:
        chunks = item['input_ids']
        masks = item['attention_mask']
        
        padded_chunks = []
        padded_masks = []
        
        # Pad existing chunks to max_len
        for chunk, mask in zip(chunks, masks):
            pad_len = max_len - len(chunk)
            padded_chunks.append(chunk + [PAD_TOKEN_ID] * pad_len)
            padded_masks.append(mask + [0] * pad_len)
            
        # Pad missing chunks with fully masked empty chunks up to max_chunks
        while len(padded_chunks) < max_chunks:
            padded_chunks.append([PAD_TOKEN_ID] * max_len)
            padded_masks.append([0] * max_len)
            
        batch_input_ids.append(padded_chunks)
        batch_attention_mask.append(padded_masks)

        # --- THE FIX ---
        # Check that the key exists AND that it is not None
        if 'labels' in item and item['labels'] is not None:
            labels.append(item['labels'])
        if 'id' in item:
            ids.append(item['id'])
            
    return {
        'input_ids': torch.tensor(batch_input_ids, dtype=torch.long),
        'attention_mask': torch.tensor(batch_attention_mask, dtype=torch.long),
        'labels': torch.tensor(labels, dtype=torch.float32) if labels else None,
        'ids': ids
    }


# Create DataLoaders (adjust batch_size based on your GPU VRAM, e.g., 4, 8, or 16)
BATCH_SIZE = 2
train_loader = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=BATCH_SIZE, collate_fn=collate_fn_chunked)
val_loader = DataLoader(tokenized_datasets["validation"], batch_size=BATCH_SIZE, collate_fn=collate_fn_chunked)
test_loader = DataLoader(tokenized_datasets["test"], batch_size=BATCH_SIZE, collate_fn=collate_fn_chunked)


# --- 2. Training Setup ---

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- ACTIVATE BOTH GPUS ---
if torch.cuda.device_count() > 1:
    print(f"Let's use {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

model.to(device)


LEARNING_RATE = 3e-4 # Higher LR is typical for PEFT compared to full fine-tuning

EPOCHS = 7
# How many epochs to actually run in this Kaggle session (set to 1 to beat the 12h limit)
EPOCHS_THIS_SESSION = 4

# Define total steps for the FULL 3-epoch journey so the LR decays correctly
total_steps = len(train_loader) * EPOCHS

optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=int(0.1 * total_steps), 
    num_training_steps=total_steps
)

# --- UPDATE: Initialize AMP GradScaler for massive speedup ---
scaler = torch.amp.GradScaler('cuda')

# Start tracking variables
start_epoch = 3
best_macro_f1 = 0.0
best_threshold=0.0

# --- 3. Training & Evaluation Loop ---

def evaluate(model, dataloader):
    model.eval()
    all_preds_probs = []
    all_labels = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # --- UPDATE: Use AMP in evaluation too ---
            with torch.amp.autocast('cuda'):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                probs = torch.sigmoid(outputs['logits'])
            
            all_preds_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    all_preds_probs = np.array(all_preds_probs)
    all_labels = np.array(all_labels)
    
    # Dynamic Thresholding to maximize Macro-F1
    best_f1 = 0
    best_thresh = 0.5
    for thresh in np.arange(0.1, 0.9, 0.05):
        binary_preds = (all_preds_probs > thresh).astype(int)
        # --- UPDATE: Added zero_division=0 to prevent sklearn spam logs ---
        current_f1 = f1_score(all_labels, binary_preds, average='macro', zero_division=0)
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_thresh = thresh
            
    return best_f1, best_thresh

print("Starting LORMA Training...")

accumulation_steps = 4 # physical batch (2) * steps (4) = effective batch (8)

checkpoint_path = "/kaggle/input/notebooks/krishnamakwana20/cs781p5s1/lorma_checkpoint.pt"

if os.path.exists(checkpoint_path):
    print("Loading checkpoint...")
    # weights_only=False required in PyTorch 2.6 to load numpy/random states
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    if hasattr(model, 'module'):
        model.module.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint['model_state_dict'])
        
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_macro_f1 = checkpoint['best_macro_f1']
    
    # Safe RNG Restoration (.cpu().byte() prevents the TypeError)
    torch.set_rng_state(checkpoint['torch_rng_state'].cpu().byte())
    cuda_states = [state.cpu().byte() for state in checkpoint['cuda_rng_state']]
    torch.cuda.set_rng_state_all(cuda_states)
    np.random.set_state(checkpoint['numpy_rng_state'])
    random.setstate(checkpoint['python_rng_state'])
    print(f"Resumed successfully from Epoch {start_epoch}")
else:
    print("No checkpoint found. Starting training from scratch.")

for epoch in range(start_epoch, start_epoch + EPOCHS_THIS_SESSION):
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    # Zero gradients at the start of the epoch
    optimizer.zero_grad() 
    
    for step, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # --- UPDATE: AMP context and Multi-GPU Loss Fix ---
        with torch.amp.autocast('cuda'):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            
            loss = outputs['loss']
            
            # If using DataParallel, loss is a vector (one per GPU). Take the mean.
            if loss.dim() > 0:
                loss = loss.mean()
                
            # Scale the loss by the accumulation steps
            loss = loss / accumulation_steps
            
        # --- UPDATE: Use Scaler for backward pass ---
        scaler.scale(loss).backward()
        
        # Only update weights after accumulating enough gradients
        if (step + 1) % accumulation_steps == 0 or (step + 1) == len(train_loader):
            # Unscale before clipping to prevent false magnitude readings
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
            
            # Step the scaler, update, and clear
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad() 
            
        # Keep track of actual loss for logging (multiply back the scaled loss)
        total_loss += loss.item() * accumulation_steps
        progress_bar.set_postfix({'loss': loss.item() * accumulation_steps})
        
    avg_train_loss = total_loss / len(train_loader)
 
    # Extract state_dict safely regardless of whether nn.DataParallel is used
    model_state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
    
    torch.save({
        'epoch': epoch,
        'model_state_dict': model_state,
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict() if 'scaler' in locals() else None, 
        'best_macro_f1': best_macro_f1,
        'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_state': torch.cuda.get_rng_state_all(), 
        'numpy_rng_state': np.random.get_state(),
        'python_rng_state': random.getstate()
    }, "lorma_checkpoint.pt")
        
    # Evaluate on validation set
    val_macro_f1, best_threshold = evaluate(model, val_loader)
    print(f"Epoch {epoch+1} - Avg Train Loss: {avg_train_loss:.4f} - Dev Macro-F1: {val_macro_f1:.4f} (Optimal Threshold: {best_threshold:.2f})")
    
    # Save the best LORMA model weights
    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        # --- UPDATE: Use `model_state` to avoid saving weights with the `module.` prefix ---
        torch.save(model_state, "best_lorma_model.pt")
        print("--> New best model saved!")

## 5. Prediction for Test cases and submission

In [ ]:
import json
import zipfile
import torch
from tqdm.auto import tqdm

print("Generating predictions for the test set...")

# 1. Load best model weights safely
best_weights = torch.load("best_lorma_model.pt", map_location=device, weights_only=True)

if hasattr(model, 'module'):
    model.module.load_state_dict(best_weights)
else:
    model.load_state_dict(best_weights)

model.eval()

# 2. Load the optimal threshold we saved during training
# If you didn't save it to a file, just hardcode the best threshold you saw in the logs (e.g., best_t = 0.85)

best_t = best_threshold
print(f"Loaded optimal threshold: {best_t}")


predictions = []

# 3. Run inference
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)      
        
        # Use AMP for inference speed and safety
        with torch.amp.autocast('cuda'):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs['logits']
            
            # Apply Sigmoid for MULTI-LABEL probabilities
            probs = torch.sigmoid(logits)
        
        # Apply the threshold to create a binary multi-hot vector (0s and 1s)
        binary_preds = (probs > best_t).int().cpu().tolist()
        
        # Extract the case IDs 
        batch_ids = batch['ids']
        
        # Now 'pred' is a list of 0s and 1s for all NUM_STATUTES
        for case_id, pred in zip(batch_ids, binary_preds):
            predictions.append({
                "id": case_id,
                "pred": pred 
            })

# 4. Save to prediction.json
output_file = "prediction.json"
with open(output_file, "w", encoding='utf-8') as f:
    json.dump(predictions, f, indent=4)

print(f"Predictions saved to {output_file}.")

# 5. Zip the file for Codabench submission
zip_filename = "submission.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # arcname ensures the file sits at the root of the zip
    zipf.write(output_file, arcname=output_file)

print(f"Successfully created {zip_filename}! You are ready to submit to Codabench.")